# Binance USD-M Perpetual Full-Market Klines

This notebook shows how to use the local `binance_klines_data_fetch` package from Jupyter, load the current Binance USD-M perpetual futures universe from `/fapi/v1/exchangeInfo`, and fetch closed 1-minute klines for the full market.

Run the cleanup cell at the end when you are done because `MultiSymbolKlineService` starts a background thread.

## 1. Add The Project To `sys.path`

In [1]:
import sys
from pathlib import Path
import numpy as np
from tqdm import tqdm

repo_path = Path("/home/suncong/binance_klines_data_fetch") ### 修改
repo_path_str = str(repo_path)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

import pandas as pd
from IPython.display import display
from binance_klines_data_fetch import (
    MultiSymbolKlineService,
    get_um_perpetual_symbol_info,
    get_um_perpetual_symbols,
)

print("Imported package from:", repo_path)

Imported package from: /home/suncong/binance_klines_data_fetch


## 2. Load Current USD-M Perpetual Symbol Universe

This cell accesses the real Binance REST API once through `/fapi/v1/exchangeInfo`. The helper keeps only symbols where `contractType == "PERPETUAL"` and `status == "TRADING"`.

In [4]:
# Set quote_assets=None for all USD-M perpetual contracts.
# Use quote_assets=["USDT"] if you only want USDT-quoted contracts.
quote_assets = ['USDT']

symbol_info = get_um_perpetual_symbol_info(quote_assets=quote_assets)
symbols = symbol_info["symbol"].tolist()

print("USD-M perpetual symbols:", len(symbols))
display(symbol_info[["symbol", "baseAsset", "quoteAsset", "marginAsset", "status"]].head(20))
print(symbols[:20])

USD-M perpetual symbols: 528


,symbol,baseAsset,quoteAsset,marginAsset,status
0,0GUSDT,0G,USDT,USDT,TRADING
1,1000000BOBUSDT,1000000BOB,USDT,USDT,TRADING
2,1000000MOGUSDT,1000000MOG,USDT,USDT,TRADING
3,1000BONKUSDT,1000BONK,USDT,USDT,TRADING
4,1000CATUSDT,1000CAT,USDT,USDT,TRADING
5,1000CHEEMSUSDT,1000CHEEMS,USDT,USDT,TRADING
6,1000FLOKIUSDT,1000FLOKI,USDT,USDT,TRADING
7,1000LUNCUSDT,1000LUNC,USDT,USDT,TRADING
8,1000PEPEUSDT,1000PEPE,USDT,USDT,TRADING
9,1000RATSUSDT,1000RATS,USDT,USDT,TRADING


['0GUSDT', '1000000BOBUSDT', '1000000MOGUSDT', '1000BONKUSDT', '1000CATUSDT', '1000CHEEMSUSDT', '1000FLOKIUSDT', '1000LUNCUSDT', '1000PEPEUSDT', '1000RATSUSDT', '1000SATSUSDT', '1000SHIBUSDT', '1000XECUSDT', '1INCHUSDT', '1MBABYDOGEUSDT', '2ZUSDT', '4USDT', 'AAVEUSDT', 'ACEUSDT', 'ACHUSDT']


## 3. Start Full-Market Background Kline Service

`window_size=20` keeps the initial full-market bootstrap modest. Increase it if you need a deeper rolling history. All symbols share one process-local weighted rate limiter.

In [6]:
service = MultiSymbolKlineService(
    symbols=symbols,
    window_size=20,
    max_workers=8,
    refresh_interval_seconds=5.0,
    startup_timeout_seconds=300.0,
)

service.start(block_until_ready=True, timeout=300.0)
print("service ready:", service.status().ready)
print("symbols loaded in service:", len(service.status().symbols))

service ready: True
symbols loaded in service: 528


In [5]:
# service.stop()

## 4. Read One Symbol

In [16]:
sample_symbol = "BTCUSDT" if "BTCUSDT" in symbols else symbols[0]
sample_df = service.get_recent(sample_symbol, 10)

# print(sample_symbol, "rows:", len(sample_df), "last_open_time:", sample_df.index[-1])
# display(sample_df.tail(20))

In [17]:
sample_df

,Open,High,Low,Close,Volume,Close_Time,Quote_Asset_Volume,Number_of_Trades,Taker_Buy_Base_Asset_Volume,Taker_Buy_Quote_Asset_Volume
Open_Time,,,,,,,,,,
2026-05-30 08:16:00+00:00,73508.2,73508.2,73501.3,73501.3,6.518,2026-05-30 08:16:59.999000+00:00,4.791102e+05,382,2.518,1.850886e+05
2026-05-30 08:17:00+00:00,73501.4,73501.4,73490.4,73490.4,15.537,2026-05-30 08:17:59.999000+00:00,1.141883e+06,385,7.967,5.855098e+05
2026-05-30 08:18:00+00:00,73490.4,73490.5,73477.2,73477.2,12.407,2026-05-30 08:18:59.999000+00:00,9.117408e+05,473,3.156,2.319181e+05
2026-05-30 08:19:00+00:00,73477.3,73477.3,73473.7,73473.8,13.682,2026-05-30 08:19:59.999000+00:00,1.005309e+06,305,3.474,2.552591e+05
2026-05-30 08:20:00+00:00,73473.7,73473.8,73473.7,73473.7,9.889,2026-05-30 08:20:59.999000+00:00,7.265822e+05,213,7.733,5.681729e+05
2026-05-30 08:21:00+00:00,73473.8,73489.1,73473.7,73489.1,21.496,2026-05-30 08:21:59.999000+00:00,1.579552e+06,555,19.871,1.460140e+06
2026-05-30 08:22:00+00:00,73489.0,73500.0,73489.0,73499.9,14.679,2026-05-30 08:22:59.999000+00:00,1.078782e+06,427,12.360,9.083470e+05
2026-05-30 08:23:00+00:00,73499.9,73500.0,73499.9,73500.0,8.309,2026-05-30 08:23:59.999000+00:00,6.107114e+05,145,7.253,5.330955e+05
2026-05-30 08:24:00+00:00,73500.0,73517.8,73499.9,73517.7,5.551,2026-05-30 08:24:59.999000+00:00,4.080409e+05,408,5.358,3.938540e+05


In [34]:
# symbols

['0GUSDT',
 '1000000BOBUSDT',
 '1000000MOGUSDT',
 '1000BONKUSDT',
 '1000CATUSDT',
 '1000CHEEMSUSDT',
 '1000FLOKIUSDT',
 '1000LUNCUSDT',
 '1000PEPEUSDT',
 '1000RATSUSDT',
 '1000SATSUSDT',
 '1000SHIBUSDT',
 '1000XECUSDT',
 '1INCHUSDT',
 '1MBABYDOGEUSDT',
 '2ZUSDT',
 '4USDT',
 'AAVEUSDT',
 'ACEUSDT',
 'ACHUSDT',
 'ACTUSDT',
 'ACUUSDT',
 'ACXUSDT',
 'ADAUSDT',
 'AERGOUSDT',
 'AEROUSDT',
 'AEVOUSDT',
 'AGLDUSDT',
 'AGTUSDT',
 'AIAUSDT',
 'AIGENSYNUSDT',
 'AINUSDT',
 'AIOTUSDT',
 'AIOUSDT',
 'AIXBTUSDT',
 'AKEUSDT',
 'AKTUSDT',
 'ALCHUSDT',
 'ALGOUSDT',
 'ALICEUSDT',
 'ALLOUSDT',
 'ALLUSDT',
 'ALPINEUSDT',
 'ALTUSDT',
 'ANIMEUSDT',
 'ANKRUSDT',
 'APEUSDT',
 'API3USDT',
 'APRUSDT',
 'APTUSDT',
 'ARBUSDT',
 'ARCUSDT',
 'ARIAUSDT',
 'ARKMUSDT',
 'ARKUSDT',
 'ARPAUSDT',
 'ARUSDT',
 'ASRUSDT',
 'ASTERUSDT',
 'ASTRUSDT',
 'ATHUSDT',
 'ATOMUSDT',
 'ATUSDT',
 'AUCTIONUSDT',
 'AUSDT',
 'AVAAIUSDT',
 'AVAUSDT',
 'AVAXUSDT',
 'AVNTUSDT',
 'AWEUSDT',
 'AXLUSDT',
 'AXSUSDT',
 'AZTECUSDT',
 'B2USDT',
 'BA

In [75]:
alpha=['4USDT',
 'ACUUSDT',
 'AIAUSDT',
 'AIGENSYNUSDT',
 'AIOUSDT',
 'AKEUSDT',
 'APRUSDT',
 'ARIAUSDT',
 'ATUSDT',
 'BASEDUSDT',
 'BASUSDT',
 'BEATUSDT',
 'BILLUSDT',
 'BIRBUSDT',
 'BLESSUSDT',
 'BLUAIUSDT',
 'BSBUSDT',
 'BULLAUSDT',
 'CARVUSDT',
 'CLANKERUSDT',
 'CLOUSDT',
 'COAIUSDT',
 'COLLECTUSDT',
 'CYSUSDT',
 'DOODUSDT',
 'ELSAUSDT',
 'ESPUSDT',
 'FIGHTUSDT',
 'FOLKSUSDT',
 'GENIUSUSDT',
 'GUAUSDT',
 'GWEIUSDT',
 'HANAUSDT',
 'INUSDT',
 'INXUSDT',
 'IRYSUSDT',
 'JCTUSDT',
 'JELLYJELLYUSDT',
 'KGENUSDT',
 'LABUSDT',
 'LYNUSDT',
 'MAGMAUSDT',
 'NIGHTUSDT',
 'ONUSDT',
 'OPGUSDT',
 'PHAROSUSDT',
 'PIEVERSEUSDT',
 'POWERUSDT',
 'PRLUSDT',
 'PTBUSDT',
 'QUSDT',
 'RAVEUSDT',
 'RIVERUSDT',
 'ROBOUSDT',
 'SAPIENUSDT',
 'SIRENUSDT',
 'SKRUSDT',
 'SPACEUSDT',
 'SPORTFUNUSDT',
 'STABLEUSDT',
 'STARUSDT',
 'TRADOORUSDT',
 'TRIAUSDT',
 'TRUTHUSDT',
 'UAIUSDT',
 'UBUSDT',
 'USUSDT',
 'WETUSDT',
 'XNYUSDT',
 'XPINUSDT',
 'ZKPUSDT']
signal_all=pd.DataFrame(columns=symbols)

signal_all.loc['sig_0_1']=np.nan
signal_all.loc['sig_0_0']=np.nan
hold_all=pd.DataFrame(columns=symbols)
hold_all.loc['amount']=0
hold_list=[]

In [126]:
# import time
# from IPython.display import clear_output
time_0=time.time()
while 1:
    if time_0<time.time():
        time_0+=30
        clear_output()
        for symboli in tqdm(symbols):
            sample_df = service.get_recent(symboli, 20)
            signal_all.loc['sig_0_0',symboli]=1000000*((sample_df['High'].rolling(10).min()-sample_df['Low'].shift(10).rolling(10).max())/(sample_df['Open'])).iloc[-1]
            signal_all.loc['amo',symboli]=sample_df['Quote_Asset_Volume'].sum()+1
            signal_all.loc['count',symboli]=sample_df['Number_of_Trades'].iloc[-1]
            signal_all.loc['sig_0_1',symboli]=signal_all.loc['count',symboli]
        new_buy_symbol=signal_all.loc['sig_0_1'][signal_all.loc['amo']>1000000].sort_values().index[0]
        hold_all.loc['amount', new_buy_symbol]+=30  
        hold_list.append(new_buy_symbol)
        
        if len(hold_list)==31:
            hold_all.loc['amount',hold_list[0]]-=30
            print('buy ',hold_list[0])
            hold_list=hold_list[1:]
        print('sell ',new_buy_symbol)
        # print(hold_all.loc['amount'].sort_values(ascending=False).iloc[:15]) 
        print(hold_list)
        print(signal_all.loc['sig_0_1'][signal_all.loc['amo']>1000000].sort_values(ascending=False)) 
        
    time.sleep(5)

    

100%|████████████████████████████████████████████████████| 528/528 [00:01<00:00, 396.22it/s]


buy  FILUSDT
sell  RENDERUSDT
['OPNUSDT', 'OPNUSDT', '1000PEPEUSDT', '1000PEPEUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'DOGEUSDT', 'DOGEUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'FILUSDT', 'ONDOUSDT', 'ONDOUSDT', 'ADAUSDT', 'RENDERUSDT']
LABUSDT         14573.0
ETHUSDT          9781.0
HUSDT            8372.0
BTCUSDT          6923.0
ALLOUSDT         6583.0
BNBUSDT          5080.0
HYPEUSDT         4579.0
WLDUSDT          3360.0
ZECUSDT          3234.0
XLMUSDT          2813.0
LTCUSDT          2713.0
BSBUSDT          2567.0
PORTALUSDT       2423.0
APRUSDT          2244.0
STGUSDT          1791.0
FETUSDT          1751.0
SOLUSDT          1635.0
SEIUSDT          1339.0
XRPUSDT          1338.0
TAOUSDT          1222.0
1000PEPEUSDT     1222.0
INJUSDT          1209.0
HEIUSDT          1120.0
DOGEUSDT         1084.0
IDUSDT            953.0
ENAUSDT           928.


KeyboardInterrupt



RAREUSDT      60
POWRUSDT       0
POWERUSDT      0
PORTALUSDT     0
POPCATUSDT     0
              ..
ESPUSDT        0
ETCUSDT        0
ETHFIUSDT      0
ETHUSDT        0
ENSOUSDT       0
Name: amount, Length: 528, dtype: int64

In [119]:
for symboli in tqdm(symbols):
            sample_df = service.get_recent(symboli, 20)
            signal_all.loc['sig_0_0',symboli]=1000000*((sample_df['High'].rolling(10).min()-sample_df['Low'].shift(10).rolling(10).max())/(sample_df['Open'])).iloc[-1]
            signal_all.loc['amo',symboli]=sample_df['Quote_Asset_Volume'].sum()+1
            signal_all.loc['count',symboli]=sample_df['Number_of_Trades'].sum()
            signal_all.loc['sig_0_1',symboli]=signal_all.loc['sig_0_0',symboli]/signal_all.loc['count',symboli]
signal_all.loc['sig_0_1'][signal_all.loc['amo']>1000000].sort_values()

100%|████████████████████████████████████████████████████| 528/528 [00:01<00:00, 403.53it/s]


PORTALUSDT     -1.145758
SKYAIUSDT      -1.085576
VTHOUSDT       -0.756342
NEARUSDT       -0.561814
BSBUSDT        -0.513154
LITUSDT        -0.360619
IDUSDT         -0.259225
HUSDT          -0.244420
HEIUSDT        -0.198833
ALLOUSDT       -0.162637
BILLUSDT       -0.134127
SUIUSDT        -0.102870
BNBUSDT        -0.093926
DOGEUSDT       -0.092194
1000PEPEUSDT   -0.073476
ZECUSDT        -0.059856
HYPEUSDT       -0.036079
XLMUSDT        -0.022225
ETHUSDT        -0.007924
BTCUSDT         0.003668
SOLUSDT         0.014000
LABUSDT         0.029766
XRPUSDT         0.043906
HBARUSDT        0.079428
FETUSDT         0.240243
WLDUSDT         0.298380
INJUSDT         0.391447
STGUSDT         0.411081
NFPUSDT         0.883361
Name: sig_0_1, dtype: float64

## 5. Build A Full-Market Latest-Candle Summary

In [8]:
all_latest = service.get_all_recent(1)

rows = []
for symbol, df in all_latest.items():
    if df.empty:
        rows.append({"symbol": symbol, "ready": False})
        continue
    latest = df.iloc[-1]
    rows.append(
        {
            "symbol": symbol,
            "ready": True,
            "last_open_time": df.index[-1],
            "Open": float(latest["Open"]),
            "High": float(latest["High"]),
            "Low": float(latest["Low"]),
            "Close": float(latest["Close"]),
            "Volume": float(latest["Volume"]),
        }
    )

latest_summary = pd.DataFrame(rows).sort_values("symbol").reset_index(drop=True)
print("ready symbols:", int(latest_summary["ready"].sum()), "/", len(latest_summary))
display(latest_summary.head(50))

ready symbols: 568 / 568


,symbol,ready,last_open_time,Open,High,Low,Close,Volume
0,0GUSDT,True,2026-05-30 08:08:00+00:00,0.430800,0.430800,0.430600,0.430700,333.00
1,1000000BOBUSDT,True,2026-05-30 08:08:00+00:00,0.014620,0.014630,0.014620,0.014630,7463.00
2,1000000MOGUSDT,True,2026-05-30 08:08:00+00:00,0.132900,0.132900,0.132900,0.132900,42.40
3,1000BONKUSDC,True,2026-05-30 08:08:00+00:00,0.005495,0.005497,0.005495,0.005497,16412.00
4,1000BONKUSDT,True,2026-05-30 08:08:00+00:00,0.005500,0.005501,0.005496,0.005497,881098.00
5,1000CATUSDT,True,2026-05-30 08:08:00+00:00,0.001885,0.001886,0.001882,0.001886,1114086.00
6,1000CHEEMSUSDT,True,2026-05-30 08:08:00+00:00,0.000622,0.000623,0.000622,0.000622,1442542.00
7,1000FLOKIUSDT,True,2026-05-30 08:08:00+00:00,0.028000,0.028000,0.027980,0.027980,47129.00
8,1000LUNCUSDT,True,2026-05-30 08:08:00+00:00,0.081270,0.081280,0.081210,0.081280,36318.00
9,1000PEPEUSDC,True,2026-05-30 08:08:00+00:00,0.003417,0.003417,0.003415,0.003415,126666.00


## 6. Inspect Service And Rate-Limit Status

In [9]:
status = service.status()

print("running:", status.running)
print("ready:", status.ready)
print("last_refresh_at:", status.last_refresh_at)
print("rate_limiter:", status.rate_limiter)

symbol_status = pd.DataFrame(
    [
        {
            "symbol": symbol,
            "ready": item.ready,
            "rows": item.row_count,
            "last_open_time": item.last_open_time,
            "error": item.last_error,
        }
        for symbol, item in status.symbols.items()
    ]
).sort_values("symbol")
display(symbol_status.head(30))
display(symbol_status[symbol_status["error"].notna()].head(20))

running: True
ready: True
last_refresh_at: 2026-05-30 08:09:36.528951+00:00
rate_limiter: RateLimiterStatus(limit=2400, effective_limit=2400, window_seconds=60.0, used_weight=568, available_weight=1832, cooldown_seconds=0.0, observed_used_weight_1m=568, last_rate_limited_at=None, last_banned_at=None)


,symbol,ready,rows,last_open_time,error
0,0GUSDT,True,20,2026-05-30 08:08:00+00:00,None
1,1000000BOBUSDT,True,20,2026-05-30 08:08:00+00:00,None
2,1000000MOGUSDT,True,20,2026-05-30 08:08:00+00:00,None
3,1000BONKUSDC,True,20,2026-05-30 08:08:00+00:00,None
4,1000BONKUSDT,True,20,2026-05-30 08:08:00+00:00,None
5,1000CATUSDT,True,20,2026-05-30 08:08:00+00:00,None
6,1000CHEEMSUSDT,True,20,2026-05-30 08:08:00+00:00,None
7,1000FLOKIUSDT,True,20,2026-05-30 08:08:00+00:00,None
8,1000LUNCUSDT,True,20,2026-05-30 08:08:00+00:00,None
9,1000PEPEUSDC,True,20,2026-05-30 08:08:00+00:00,None


,symbol,ready,rows,last_open_time,error


## 7. Stop Background Thread

Run this before closing the notebook or before creating another service instance.

In [ ]:
service.stop()
print("service running:", service.status().running)